## Data Scraping

In [1]:
pip install selenium

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.5 MB 7.5 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.5 MB 7.0 MB/s eta 0:00:01
   ---------------- ----------------------- 3.9/9.5 MB 6.8 MB/s eta 0:00:01
   ------------------------ --------------- 5.8/9.5 MB 7.4 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.5 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 6.7 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

  Attempting uninstall: urllib3

    Found existing installation: urllib3 2.3.0

   ----- ---------------------------------- 1/8 [urllib3]
    Uninstalling urllib3-2.3.0:
   ----- ---------------------------------- 1/8 [urllib3]
      Successfully uninstalled 

In [1]:
from selenium import webdriver

driver = webdriver.Chrome()
driver.get("https://quotes.toscrape.com/")

In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

all_titles = []
all_links = []
all_storylines = []


for start in [1, 51, 101]:

    url = f"https://www.imdb.com/search/title/?year=2024&start={start}"
    driver.get(url)
    time.sleep(3)

    title_elements = driver.find_elements(By.CLASS_NAME, "ipc-title__text")

    for t in title_elements:
        text = t.text
        if text == "Advanced title search":
            continue
        if ". " in text:
            text = text.split(". ", 1)[1]
        all_titles.append(text)

  
    link_elements = driver.find_elements(By.CLASS_NAME, "ipc-title-link-wrapper")
    for elem in link_elements[:50]:
        all_links.append(elem.get_attribute("href"))


for link in all_links[:150]:
    driver.get(link)
    time.sleep(2)

    try:
        storyline_element = driver.find_element(
            By.CSS_SELECTOR,
            "[data-testid='plot']"
        )
        storyline = storyline_element.text
    except:
        storyline = "Not Available"

    all_storylines.append(storyline)

driver.quit()


df = pd.DataFrame({
    "Movie Name": all_titles[:150],
    "Storyline": all_storylines
})

df.to_csv("imdb_150_movies_2024.csv", index=False)

print("150 Movies Scraped Successfully")
print(df.head())

150 Movies Scraped Successfully
       Movie Name                                          Storyline
0         Fallout  In a future, post-apocalyptic Los Angeles brou...
1           Cross  Series adaptation of James Patterson novels ab...
2         Landman  A modern-day tale of fortune seeking in the wo...
3  High Potential  A single mom with three kids helps solve an un...
4   The Substance  A fading celebrity takes a black-market drug: ...


## Data Understanding

In [18]:
import pandas as pd

df = pd.read_csv("imdb_150_movies_2024.csv")

print(df.shape)
df.head()

(150, 2)


,Movie Name,Storyline
0,Fallout,"In a future, post-apocalyptic Los Angeles brou..."
1,Cross,Series adaptation of James Patterson novels ab...
2,Landman,A modern-day tale of fortune seeking in the wo...
3,High Potential,A single mom with three kids helps solve an un...
4,The Substance,A fading celebrity takes a black-market drug: ...


In [33]:
df = df[df["Movie Name"] != "Recently viewed"]
df = df.reset_index(drop=True)

print(df.shape)

(148, 3)


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Movie Name       148 non-null    object
 1   Storyline        148 non-null    object
 2   Clean_Storyline  148 non-null    object
dtypes: object(3)
memory usage: 3.6+ KB


In [35]:
print(df.isnull().sum())

Movie Name         0
Storyline          0
Clean_Storyline    0
dtype: int64


In [36]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


## NLP Pre Processing

In [37]:
import re

def clean_text(text):
    text = text.lower()  # convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation & numbers
    return text

df["Clean_Storyline"] = df["Storyline"].apply(clean_text)

df.head()

,Movie Name,Storyline,Clean_Storyline
0,Fallout,"In a future, post-apocalyptic Los Angeles brou...",in a future postapocalyptic los angeles brough...
1,Cross,Series adaptation of James Patterson novels ab...,series adaptation of james patterson novels ab...
2,Landman,A modern-day tale of fortune seeking in the wo...,a modernday tale of fortune seeking in the wor...
3,High Potential,A single mom with three kids helps solve an un...,a single mom with three kids helps solve an un...
4,The Substance,A fading celebrity takes a black-market drug: ...,a fading celebrity takes a blackmarket drug a ...


In [38]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [39]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in stop_words]
    return " ".join(filtered)

df["Clean_Storyline"] = df["Clean_Storyline"].apply(remove_stopwords)

df.head()

,Movie Name,Storyline,Clean_Storyline
0,Fallout,"In a future, post-apocalyptic Los Angeles brou...",future postapocalyptic los angeles brought nuc...
1,Cross,Series adaptation of James Patterson novels ab...,series adaptation james patterson novels compl...
2,Landman,A modern-day tale of fortune seeking in the wo...,modernday tale fortune seeking world west texa...
3,High Potential,A single mom with three kids helps solve an un...,single mom three kids helps solve unsolvable c...
4,The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity takes blackmarket drug cellre...


In [40]:
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized)

df["Clean_Storyline"] = df["Clean_Storyline"].apply(lemmatize_text)

df.head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,Movie Name,Storyline,Clean_Storyline
0,Fallout,"In a future, post-apocalyptic Los Angeles brou...",future postapocalyptic los angeles brought nuc...
1,Cross,Series adaptation of James Patterson novels ab...,series adaptation james patterson novel compli...
2,Landman,A modern-day tale of fortune seeking in the wo...,modernday tale fortune seeking world west texa...
3,High Potential,A single mom with three kids helps solve an un...,single mom three kid help solve unsolvable cri...
4,The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity take blackmarket drug cellrep...


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

tfidf_matrix = tfidf.fit_transform(df["Clean_Storyline"])

print(tfidf_matrix.shape)

(148, 650)


In [42]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

print(cosine_sim.shape)

(148, 148)


In [47]:
def recommend_by_storyline(user_input, top_n=5):
    
    text = user_input.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    
    cleaned_input = " ".join(words)
    
    input_vector = tfidf.transform([cleaned_input])
    
    similarity_scores = cosine_similarity(input_vector, tfidf_matrix)
    
    similarity_scores = list(enumerate(similarity_scores[0]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    top_movies = similarity_scores[:top_n]
    movie_indices = [i[0] for i in top_movies]
    
    return df["Movie Name"].iloc[movie_indices].reset_index(drop=True)

In [48]:
recommend_by_storyline("post apocalyptic nuclear survival world")

0             Fallout
1           Civil War
2    I Was a Stranger
3     Brilliant Minds
4             Reunion
Name: Movie Name, dtype: object